In [ ]:
# HotpotQA Step 0: Mount Drive

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/421-Project/HotpotQA")
print("Project dir exists:", PROJECT_DIR.exists())

Mounted at /content/drive
Project dir exists: True


In [ ]:
# HotpotQA Step 1: Download and inspect the dataset

import json
import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")

# Download HotpotQA distractor setting (dev set)
# This is the standard setting used in most papers
!wget -q -O "{PROJECT_DIR}/hotpot_dev_distractor_v1.json" "http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json"

# Download train set
!wget -q -O "{PROJECT_DIR}/hotpot_train_v1.1.json" "http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_train_v1.1.json"

# Check file sizes
for name in ["hotpot_dev_distractor_v1.json", "hotpot_train_v1.1.json"]:
    path = PROJECT_DIR / name
    if path.exists():
        print(f"{name}: {path.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"{name}: NOT FOUND")

# Inspect structure
with open(PROJECT_DIR / "hotpot_dev_distractor_v1.json", "r") as f:
    hotpot_dev = json.load(f)

print(f"\nDev set size: {len(hotpot_dev)}")
print(f"Example keys: {sorted(hotpot_dev[0].keys())}")

# Look at 2 examples
for i in range(2):
    ex = hotpot_dev[i]
    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print(f"Question: {ex['question']}")
    print(f"Answer: {ex['answer']}")
    print(f"Type: {ex.get('type', 'N/A')}")
    print(f"Level: {ex.get('level', 'N/A')}")
    print(f"Num contexts: {len(ex.get('context', []))}")
    if ex.get('context'):
        print(f"Context[0] title: {ex['context'][0][0]}")
        print(f"Context[0] num sentences: {len(ex['context'][0][1])}")
    print(f"Supporting facts: {ex.get('supporting_facts', [])[:5]}...")

hotpot_dev_distractor_v1.json: 46.3 MB
hotpot_train_v1.1.json: 566.4 MB

Dev set size: 7405
Example keys: ['_id', 'answer', 'context', 'level', 'question', 'supporting_facts', 'type']

Example 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Type: comparison
Level: hard
Num contexts: 10
Context[0] title: Ed Wood (film)
Context[0] num sentences: 3
Supporting facts: [['Scott Derrickson', 0], ['Ed Wood', 0]]...

Example 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Answer: Chief of Protocol
Type: bridge
Level: hard
Num contexts: 10
Context[0] title: Meet Corliss Archer
Context[0] num sentences: 4
Supporting facts: [['Kiss and Tell (1945 film)', 0], ['Shirley Temple', 0], ['Shirley Temple', 1]]...


In [ ]:
# HotpotQA Step 2: Build helper functions

!pip -q install rank_bm25

import json
import re
from collections import Counter
from rank_bm25 import BM25Okapi

# Re-define shared utilities from MuSiQue pipeline
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def simple_tokenize(text):
    text = text.lower()
    return re.findall(r"\w+", text)

# Load data
with open(PROJECT_DIR / "hotpot_dev_distractor_v1.json", "r") as f:
    hotpot_dev = json.load(f)

print(f"Dev set: {len(hotpot_dev)} examples")

type_counts = Counter(ex["type"] for ex in hotpot_dev)
level_counts = Counter(ex["level"] for ex in hotpot_dev)
print(f"Types: {dict(type_counts)}")
print(f"Levels: {dict(level_counts)}")

def hotpot_context_to_text(context_item):
    title, sentences = context_item
    text = " ".join(sentences)
    return f"Title: {title}\n{text}"

def build_hotpot_all_context(ex):
    return "\n\n".join(hotpot_context_to_text(c) for c in ex["context"])

def build_hotpot_supporting_context(ex):
    supporting_titles = set(title for title, _ in ex["supporting_facts"])
    supporting_paras = [c for c in ex["context"] if c[0] in supporting_titles]
    return "\n\n".join(hotpot_context_to_text(c) for c in supporting_paras)

def build_hotpot_bm25_topk(ex, k=5):
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    tokenized = [simple_tokenize(t) for t in para_texts]
    bm25 = BM25Okapi(tokenized)
    query_tokens = simple_tokenize(ex["question"])
    scores = bm25.get_scores(query_tokens)
    ranked = sorted(zip(ex["context"], para_texts, scores), key=lambda x: x[2], reverse=True)
    topk = ranked[:k]
    context = "\n\n".join(t for _, t, _ in topk)
    supporting_titles = set(title for title, _ in ex["supporting_facts"])
    retrieved_titles = set(c[0] for c, _, _ in topk)
    recall = len(supporting_titles & retrieved_titles) / len(supporting_titles) if supporting_titles else 0
    full_recall = int(supporting_titles <= retrieved_titles)
    return context, recall, full_recall

yn_count = sum(1 for ex in hotpot_dev if ex["answer"].lower() in ["yes", "no"])
print(f"\nYes/No answers: {yn_count} ({yn_count/len(hotpot_dev)*100:.1f}%)")

# Verify helpers
ex = hotpot_dev[1]
print(f"\nTest example:")
print(f"Question: {ex['question']}")
print(f"Answer: {ex['answer']}")
print(f"Type: {ex['type']}")
print(f"\nSupporting context preview:")
print(build_hotpot_supporting_context(ex)[:500])
print(f"\nBM25 top-3 recall:")
_, recall, full = build_hotpot_bm25_topk(ex, k=3)
print(f"  Partial recall: {recall:.2f}, Full recall: {full}")

Dev set: 7405 examples
Types: {'comparison': 1487, 'bridge': 5918}
Levels: {'hard': 7405}

Yes/No answers: 458 (6.2%)

Test example:
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Answer: Chief of Protocol
Type: bridge

Supporting context preview:
Title: Shirley Temple
Shirley Temple Black (April 23, 1928 – February 10, 2014) was an American actress, singer, dancer, businesswoman, and diplomat who was Hollywood's number one box-office draw as a child actress from 1935 to 1938.  As an adult, she was named United States ambassador to Ghana and to Czechoslovakia and also served as Chief of Protocol of the United States.

Title: Kiss and Tell (1945 film)
Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as 

BM25 top-3 recall:
  Partial recall: 0.50, Full recall: 0


In [ ]:
# HotpotQA Step 3: Load model

!pip -q install transformers accelerate sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-xxl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cuda
Memory: 26.6 GB


In [ ]:
import torch
import subprocess

# Check GPU info
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

print(f"PyTorch sees: {torch.cuda.get_device_name(0)}")
print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Currently allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Fri Apr 17 16:38:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             48W /  400W |   25792MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import torch
import gc

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"After cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")
print(f"Reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB reserved")

After cleanup: 40.6 GB allocated
Reserved: 41.7 GB reserved


In [ ]:
# HotpotQA Step 4: Define prompts and helpers

import time
from collections import defaultdict

def build_qa_prompt(question, context):
    return f"""Answer the question using the context below.

Question: {question}

Context:
{context}

Answer:"""

def build_cot_prompt_v3(question, context):
    return f"""Answer the question using the context below.

Reason step by step, then on a new line write only "Answer:" followed by the short final answer.

Question: {question}

Context:
{context}

Reasoning:"""

def extract_answer_v3(output):
    import re
    matches = list(re.finditer(r"(?:Final\s+)?Answer:\s*(.*)", output, re.IGNORECASE))
    if matches:
        return matches[-1].group(1).strip().rstrip(".")

    match = re.search(r"(?:so,?\s+)?the\s+(?:final\s+)?answer\s+is\s+(.*)", output, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip(".")

    first_sent = re.split(r'[.\n]', output.strip())[0].strip()
    return first_sent if first_sent else output.strip()

def get_qtype(ex):
    return ex.get("type", "unknown")

def run_model(prompt, max_tokens=32):
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=4096
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            num_beams=4, early_stopping=True,
            no_repeat_ngram_size=3 if max_tokens > 32 else 0
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("Helpers defined.")
print(f"Dev set: {len(hotpot_dev)} examples")
print(f"Types: bridge={sum(1 for e in hotpot_dev if e['type']=='bridge')}, "
      f"comparison={sum(1 for e in hotpot_dev if e['type']=='comparison')}")

Helpers defined.
Dev set: 7405 examples
Types: bridge=5918, comparison=1487


In [ ]:
# HotpotQA B1: All paragraphs

import time

total = len(hotpot_dev)
em_scores = defaultdict(list)
f1_scores = defaultdict(list)

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    context = build_hotpot_all_context(ex)
    prompt = build_qa_prompt(ex["question"], context)
    pred = run_model(prompt)

    em_scores[qtype].append(compute_em(pred, ex["answer"]))
    f1_scores[qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        all_em = [s for q in em_scores for s in em_scores[q]]
        elapsed = time.time() - start
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} Time={elapsed:.0f}s")

# Results
print("\n" + "=" * 80)
print(f"{'B1: All Paragraphs':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

all_em = [s for q in em_scores for s in em_scores[q]]
all_f1 = [s for q in f1_scores for s in f1_scores[q]]
bridge_em = sum(em_scores["bridge"]) / len(em_scores["bridge"])
comp_em = sum(em_scores["comparison"]) / len(em_scores["comparison"])
total_em = sum(all_em) / len(all_em)
total_f1 = sum(all_f1) / len(all_f1)

print(f"{'B1: All Paragraphs':<25} {bridge_em:>12.3f} {comp_em:>12.3f} {total_em:>12.3f} {total_f1:>12.3f}")
print("=" * 80)

[500/7405] EM=0.512 Time=420s
[1000/7405] EM=0.509 Time=836s
[1500/7405] EM=0.495 Time=1255s
[2000/7405] EM=0.495 Time=1676s
[2500/7405] EM=0.494 Time=2093s
[3000/7405] EM=0.490 Time=2525s
[3500/7405] EM=0.489 Time=2941s
[4000/7405] EM=0.486 Time=3357s
[4500/7405] EM=0.483 Time=3784s
[5000/7405] EM=0.483 Time=4202s
[5500/7405] EM=0.485 Time=4621s
[6000/7405] EM=0.485 Time=5044s
[6500/7405] EM=0.487 Time=5465s
[7000/7405] EM=0.485 Time=5879s

B1: All Paragraphs           Bridge EM      Comp EM       All EM       All F1
B1: All Paragraphs               0.455        0.608        0.486        0.660


In [ ]:
# Verify HotpotQA training set structure

with open(PROJECT_DIR / "hotpot_train_v1.1.json", "r") as f:
    hotpot_train = json.load(f)

print(f"Train set size: {len(hotpot_train)}")
print(f"Keys: {sorted(hotpot_train[0].keys())}")

ex = hotpot_train[0]
print(f"\nExample question: {ex['question']}")
print(f"Answer: {ex['answer']}")
print(f"Type: {ex['type']}")
print(f"Num context paragraphs: {len(ex['context'])}")
print(f"Supporting facts: {ex['supporting_facts'][:5]}")
print(f"\nContext[0]: title='{ex['context'][0][0]}', "
      f"num_sentences={len(ex['context'][0][1])}")
print(f"Context[1]: title='{ex['context'][1][0]}', "
      f"num_sentences={len(ex['context'][1][1])}")

Train set size: 90447
Keys: ['_id', 'answer', 'context', 'level', 'question', 'supporting_facts', 'type']

Example question: Which magazine was started first Arthur's Magazine or First for Women?
Answer: Arthur's Magazine
Type: comparison
Num context paragraphs: 10
Supporting facts: [["Arthur's Magazine", 0], ['First for Women', 0]]

Context[0]: title='Radio City (Indian radio station)', num_sentences=7
Context[1]: title='History of Albanian football', num_sentences=4


In [ ]:
# HotpotQA B2: BM25 Top-5

import time

total = len(hotpot_dev)
em_scores_b2 = defaultdict(list)
f1_scores_b2 = defaultdict(list)
recall_partial = []
full_recall_count = 0

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)

    context, partial_rec, full_rec = build_hotpot_bm25_topk(ex, k=5)
    prompt = build_qa_prompt(ex["question"], context)
    pred = run_model(prompt)

    em_scores_b2[qtype].append(compute_em(pred, ex["answer"]))
    f1_scores_b2[qtype].append(compute_f1(pred, ex["answer"]))
    recall_partial.append(partial_rec)
    full_recall_count += full_rec

    if i % 500 == 0:
        all_em = [s for q in em_scores_b2 for s in em_scores_b2[q]]
        elapsed = time.time() - start
        avg_recall = sum(recall_partial) / len(recall_partial)
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} "
              f"Recall={avg_recall:.3f} Time={elapsed:.0f}s")

print("\n" + "=" * 80)
print(f"{'B2: BM25 Top-5':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

all_em = [s for q in em_scores_b2 for s in em_scores_b2[q]]
all_f1 = [s for q in f1_scores_b2 for s in f1_scores_b2[q]]
bridge_em = sum(em_scores_b2["bridge"]) / len(em_scores_b2["bridge"])
comp_em = sum(em_scores_b2["comparison"]) / len(em_scores_b2["comparison"])

print(f"{'B2: BM25 Top-5':<25} {bridge_em:>12.3f} {comp_em:>12.3f} "
      f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")
print("=" * 80)
print(f"\nRetrieval Recall@5: {sum(recall_partial)/len(recall_partial):.3f} (partial)")
print(f"Full Recall@5: {full_recall_count/total:.3f} ({full_recall_count}/{total})")

[500/7405] EM=0.424 Recall=0.804 Time=242s
[1000/7405] EM=0.437 Recall=0.816 Time=488s
[1500/7405] EM=0.422 Recall=0.812 Time=729s
[2000/7405] EM=0.418 Recall=0.808 Time=971s
[2500/7405] EM=0.420 Recall=0.810 Time=1214s
[3000/7405] EM=0.415 Recall=0.806 Time=1461s
[3500/7405] EM=0.414 Recall=0.808 Time=1703s
[4000/7405] EM=0.416 Recall=0.807 Time=1944s
[4500/7405] EM=0.413 Recall=0.804 Time=2188s
[5000/7405] EM=0.412 Recall=0.802 Time=2426s
[5500/7405] EM=0.412 Recall=0.801 Time=2671s
[6000/7405] EM=0.410 Recall=0.800 Time=2913s
[6500/7405] EM=0.413 Recall=0.801 Time=3152s
[7000/7405] EM=0.412 Recall=0.800 Time=3389s

B2: BM25 Top-5               Bridge EM      Comp EM       All EM       All F1
B2: BM25 Top-5                   0.368        0.592        0.413        0.564

Retrieval Recall@5: 0.800 (partial)
Full Recall@5: 0.623 (4610/7405)


In [ ]:
# HotpotQA B3: Oracle (gold supporting paragraphs)

import time

total = len(hotpot_dev)
em_scores_b3 = defaultdict(list)
f1_scores_b3 = defaultdict(list)

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    context = build_hotpot_supporting_context(ex)
    prompt = build_qa_prompt(ex["question"], context)
    pred = run_model(prompt)

    em_scores_b3[qtype].append(compute_em(pred, ex["answer"]))
    f1_scores_b3[qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        all_em = [s for q in em_scores_b3 for s in em_scores_b3[q]]
        all_f1 = [s for q in f1_scores_b3 for s in f1_scores_b3[q]]
        elapsed = time.time() - start
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} "
              f"F1={sum(all_f1)/len(all_f1):.3f} Time={elapsed:.0f}s")

print("\n" + "=" * 80)
print(f"{'B3: Oracle':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

all_em = [s for q in em_scores_b3 for s in em_scores_b3[q]]
all_f1 = [s for q in f1_scores_b3 for s in f1_scores_b3[q]]
bridge_em = sum(em_scores_b3["bridge"]) / len(em_scores_b3["bridge"])
comp_em = sum(em_scores_b3["comparison"]) / len(em_scores_b3["comparison"])

print(f"{'B3: Oracle':<25} {bridge_em:>12.3f} {comp_em:>12.3f} "
      f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")
print("=" * 80)

[500/7405] EM=0.554 F1=0.753 Time=165s
[1000/7405] EM=0.544 F1=0.747 Time=332s
[1500/7405] EM=0.543 F1=0.742 Time=501s
[2000/7405] EM=0.545 F1=0.741 Time=668s
[2500/7405] EM=0.546 F1=0.740 Time=840s
[3000/7405] EM=0.549 F1=0.741 Time=1012s
[3500/7405] EM=0.546 F1=0.740 Time=1180s
[4000/7405] EM=0.545 F1=0.740 Time=1347s
[4500/7405] EM=0.548 F1=0.741 Time=1513s
[5000/7405] EM=0.548 F1=0.741 Time=1677s
[5500/7405] EM=0.546 F1=0.739 Time=1847s
[6000/7405] EM=0.546 F1=0.739 Time=2017s
[6500/7405] EM=0.548 F1=0.742 Time=2183s
[7000/7405] EM=0.547 F1=0.741 Time=2348s

B3: Oracle                   Bridge EM      Comp EM       All EM       All F1
B3: Oracle                       0.518        0.668        0.548        0.741


In [ ]:
# HotpotQA CoT zero-shot (oracle context)

import time
import re

total = len(hotpot_dev)
em_scores_cot0 = defaultdict(list)
f1_scores_cot0 = defaultdict(list)

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    context = build_hotpot_supporting_context(ex)
    prompt = build_cot_prompt_v3(ex["question"], context)
    raw = run_model(prompt, max_tokens=256)
    pred = extract_answer_v3(raw)

    em_scores_cot0[qtype].append(compute_em(pred, ex["answer"]))
    f1_scores_cot0[qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        all_em = [s for q in em_scores_cot0 for s in em_scores_cot0[q]]
        all_f1 = [s for q in f1_scores_cot0 for s in f1_scores_cot0[q]]
        elapsed = time.time() - start
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} "
              f"F1={sum(all_f1)/len(all_f1):.3f} Time={elapsed:.0f}s")

print("\n" + "=" * 80)
print(f"{'CoT 0-shot oracle':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

all_em = [s for q in em_scores_cot0 for s in em_scores_cot0[q]]
all_f1 = [s for q in f1_scores_cot0 for s in f1_scores_cot0[q]]
bridge_em = sum(em_scores_cot0["bridge"]) / len(em_scores_cot0["bridge"])
comp_em = sum(em_scores_cot0["comparison"]) / len(em_scores_cot0["comparison"])

print(f"{'CoT 0-shot oracle':<25} {bridge_em:>12.3f} {comp_em:>12.3f} "
      f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")
print("=" * 80)

[500/7405] EM=0.594 F1=0.756 Time=301s
[1000/7405] EM=0.607 F1=0.770 Time=606s
[1500/7405] EM=0.598 F1=0.762 Time=929s
[2000/7405] EM=0.602 F1=0.760 Time=1232s
[2500/7405] EM=0.600 F1=0.758 Time=1539s
[3000/7405] EM=0.596 F1=0.756 Time=1840s
[3500/7405] EM=0.594 F1=0.756 Time=2155s
[4000/7405] EM=0.592 F1=0.756 Time=2468s
[4500/7405] EM=0.594 F1=0.759 Time=2786s
[5000/7405] EM=0.587 F1=0.755 Time=3099s
[5500/7405] EM=0.586 F1=0.754 Time=3412s
[6000/7405] EM=0.588 F1=0.756 Time=3728s
[6500/7405] EM=0.588 F1=0.756 Time=4027s
[7000/7405] EM=0.587 F1=0.756 Time=4331s

CoT 0-shot oracle            Bridge EM      Comp EM       All EM       All F1
CoT 0-shot oracle                0.559        0.693        0.586        0.754


In [ ]:
# HotpotQA CoT few-shot (2 demos from HotpotQA training set)

# Load 2 training demos from HotpotQA train set
with open(PROJECT_DIR / "hotpot_train_v1.1.json", "r") as f:
    hotpot_train = json.load(f)

# Pick 2 good demos — one bridge, one comparison
demo_bridge = next(ex for ex in hotpot_train if ex["type"] == "bridge")
demo_comp = next(ex for ex in hotpot_train if ex["type"] == "comparison")

demo_bridge_ctx = build_hotpot_supporting_context(demo_bridge)
demo_comp_ctx = build_hotpot_supporting_context(demo_comp)

def build_hotpot_fewshot_cot_prompt(question, context):
    return f"""Answer the question using the context. Reason step by step, then on a new line write only "Answer:" followed by the short final answer.

Question: {demo_bridge["question"]}

Context:
{demo_bridge_ctx}

Reasoning: The question asks about {demo_bridge["question"].lower()[:60]}. Based on the context, I can trace the reasoning step by step to find the answer.
Answer: {demo_bridge["answer"]}

Question: {demo_comp["question"]}

Context:
{demo_comp_ctx}

Reasoning: The question compares two things. Based on the context, I can identify the relevant facts for each and determine the answer.
Answer: {demo_comp["answer"]}

Question: {question}

Context:
{context}

Reasoning:"""

# Verify prompt on one example
ex = hotpot_dev[1]
prompt = build_hotpot_fewshot_cot_prompt(ex["question"], build_hotpot_supporting_context(ex))
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)
print(f"Demo bridge Q: {demo_bridge['question'][:70]}")
print(f"Demo bridge A: {demo_bridge['answer']}")
print(f"Demo comp Q: {demo_comp['question'][:70]}")
print(f"Demo comp A: {demo_comp['answer']}")
print(f"\nPrompt token length for example: {inputs['input_ids'].shape[1]}")
print(f"Fits in 4096 tokens: {inputs['input_ids'].shape[1] < 4096}")


Demo bridge Q: The Oberoi family is part of a hotel company that has a head office in
Demo bridge A: Delhi
Demo comp Q: Which magazine was started first Arthur's Magazine or First for Women?
Demo comp A: Arthur's Magazine

Prompt token length for example: 613
Fits in 4096 tokens: True


In [ ]:
# HotpotQA CoT few-shot (2 demos, oracle context) — full dev set

import time

total = len(hotpot_dev)
em_scores_cot2 = defaultdict(list)
f1_scores_cot2 = defaultdict(list)

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    context = build_hotpot_supporting_context(ex)
    prompt = build_hotpot_fewshot_cot_prompt(ex["question"], context)
    raw = run_model(prompt, max_tokens=256)
    pred = extract_answer_v3(raw)

    em_scores_cot2[qtype].append(compute_em(pred, ex["answer"]))
    f1_scores_cot2[qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        all_em = [s for q in em_scores_cot2 for s in em_scores_cot2[q]]
        all_f1 = [s for q in f1_scores_cot2 for s in f1_scores_cot2[q]]
        elapsed = time.time() - start
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} "
              f"F1={sum(all_f1)/len(all_f1):.3f} Time={elapsed:.0f}s")

print("\n" + "=" * 80)
print(f"{'CoT 2-shot oracle':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

all_em = [s for q in em_scores_cot2 for s in em_scores_cot2[q]]
all_f1 = [s for q in f1_scores_cot2 for s in f1_scores_cot2[q]]
bridge_em = sum(em_scores_cot2["bridge"]) / len(em_scores_cot2["bridge"])
comp_em = sum(em_scores_cot2["comparison"]) / len(em_scores_cot2["comparison"])

print(f"{'CoT 2-shot oracle':<25} {bridge_em:>12.3f} {comp_em:>12.3f} "
      f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")
print("=" * 80)

print(f"\nComparison — CoT 0-shot oracle: EM=0.586, F1=0.754")

[500/7405] EM=0.622 F1=0.782 Time=400s
[1000/7405] EM=0.630 F1=0.788 Time=813s
[1500/7405] EM=0.615 F1=0.780 Time=1225s
[2000/7405] EM=0.613 F1=0.776 Time=1626s
[2500/7405] EM=0.612 F1=0.773 Time=2041s
[3000/7405] EM=0.608 F1=0.770 Time=2439s
[3500/7405] EM=0.608 F1=0.770 Time=2839s
[4000/7405] EM=0.605 F1=0.770 Time=3249s
[4500/7405] EM=0.608 F1=0.773 Time=3660s
[5000/7405] EM=0.602 F1=0.770 Time=4074s
[5500/7405] EM=0.599 F1=0.767 Time=4482s
[6000/7405] EM=0.601 F1=0.768 Time=4881s
[6500/7405] EM=0.601 F1=0.769 Time=5301s
[7000/7405] EM=0.601 F1=0.769 Time=5693s

CoT 2-shot oracle            Bridge EM      Comp EM       All EM       All F1
CoT 2-shot oracle                0.578        0.689        0.600        0.768

Comparison — CoT 0-shot oracle: EM=0.586, F1=0.754


In [ ]:
# HotpotQA: Error analysis + type-stratified breakdown
# Uses predictions from CoT 2-shot (best system)

import time
from collections import defaultdict

total = len(hotpot_dev)

# Storage for error categories by type
error_cats = {
    "Granularity mismatch": defaultdict(int),
    "Near-correct / alias gap": defaultdict(int),
    "Verbosity / extraction": defaultdict(int),
    "Partial overlap": defaultdict(int),
    "Complete miss": defaultdict(int),
    "Other": defaultdict(int),
}

total_correct = 0
total_errors = 0
type_totals = defaultdict(int)

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    type_totals[qtype] += 1

    context = build_hotpot_supporting_context(ex)
    prompt = build_hotpot_fewshot_cot_prompt(ex["question"], context)
    raw = run_model(prompt, max_tokens=256)
    pred = extract_answer_v3(raw)

    gold = ex["answer"]
    em = compute_em(pred, gold)
    f1 = compute_f1(pred, gold)

    if em == 1:
        total_correct += 1
        continue

    total_errors += 1
    norm_pred = normalize_text(pred)
    norm_gold = normalize_text(gold)

    if f1 >= 0.5 and (norm_gold in norm_pred or norm_pred in norm_gold):
        error_cats["Granularity mismatch"][qtype] += 1
    elif f1 >= 0.75:
        error_cats["Near-correct / alias gap"][qtype] += 1
    elif len(pred) > 2 * len(gold) and norm_gold in norm_pred:
        error_cats["Verbosity / extraction"][qtype] += 1
    elif 0 < f1 < 0.5:
        error_cats["Partial overlap"][qtype] += 1
    elif f1 == 0:
        error_cats["Complete miss"][qtype] += 1
    else:
        error_cats["Other"][qtype] += 1

    if i % 500 == 0:
        elapsed = time.time() - start
        print(f"[{i}/{total}] errors={total_errors} Time={elapsed:.0f}s")

# Print results
print(f"\nTotal: {total} | Correct: {total_correct} ({total_correct/total*100:.1f}%) | "
      f"Errors: {total_errors} ({total_errors/total*100:.1f}%)")

print(f"\n{'Category':<35} {'Count':>7} {'% Err':>8} {'Bridge':>8} {'Comp':>8}")
print("=" * 75)

for cat, counts in error_cats.items():
    count = sum(counts.values())
    pct = count / total_errors * 100 if total_errors > 0 else 0
    print(f"{cat:<35} {count:>7} {pct:>7.1f}% "
          f"{counts['bridge']:>8} {counts['comparison']:>8}")

print("=" * 75)
print(f"{'Total errors':<35} {total_errors:>7}")

# Type-stratified breakdown summary
print(f"\nType-stratified summary (CoT 2-shot oracle):")
print(f"  Bridge: {sum(em_scores_cot2['bridge'])/len(em_scores_cot2['bridge']):.3f} EM "
      f"({len(em_scores_cot2['bridge'])} examples)")
print(f"  Comparison: {sum(em_scores_cot2['comparison'])/len(em_scores_cot2['comparison']):.3f} EM "
      f"({len(em_scores_cot2['comparison'])} examples)")

# Compare error rates by type
bridge_errors = sum(error_cats[c]["bridge"] for c in error_cats)
comp_errors = sum(error_cats[c]["comparison"] for c in error_cats)
print(f"\n  Bridge error rate: {bridge_errors/type_totals['bridge']*100:.1f}%")
print(f"  Comparison error rate: {comp_errors/type_totals['comparison']*100:.1f}%")

[1000/7405] errors=370 Time=809s
[2500/7405] errors=971 Time=2049s
[3000/7405] errors=1177 Time=2448s
[3500/7405] errors=1372 Time=2848s
[4000/7405] errors=1581 Time=3257s
[5500/7405] errors=2203 Time=4485s
[6000/7405] errors=2397 Time=4883s
[7000/7405] errors=2794 Time=5696s

Total: 7405 | Correct: 4446 (60.0%) | Errors: 2959 (40.0%)

Category                              Count    % Err   Bridge     Comp
Granularity mismatch                   1278    43.2%     1148      130
Near-correct / alias gap                149     5.0%      118       31
Verbosity / extraction                  128     4.3%      116       12
Partial overlap                         401    13.6%      349       52
Complete miss                           870    29.4%      648      222
Other                                   133     4.5%      118       15
Total errors                           2959

Type-stratified summary (CoT 2-shot oracle):
  Bridge: 0.578 EM (5918 examples)
  Comparison: 0.689 EM (1487 examples)

